# DA3 GPU gate (Colab)
Runbook: `specs/2026-10-11-da3-gpu-gate/gpu-validation-runbook.md` Path A.
Needs: GPU runtime (T4/L4), `da3-gate.zip` + pre-exported ONNX uploaded to `/content`.
Export at home first (ORT smoke + TRT fixed-size + square image):
`python export_onnx.py --model-name da3-small --output da3-small.onnx`
`python export_onnx.py --model-name da3-small --output da3-small-fixed.onnx --fixed-size`
TensorRT 11.2.1.2 / CUDA 13.3 — same pin as rf-detr-cpp-inference.


In [ ]:
!nvidia-smi
!bash scripts/colab/setup_colab.sh


In [ ]:
!rm -rf /content/da3 && mkdir -p /content/da3 && cd /content/da3 && unzip -q /content/da3-gate.zip
!wget -q https://raw.githubusercontent.com/ByteDance-Seed/manip-as-in-sim-suite/main/cdm/example_data/color_12.png -O /content/da3/test.jpg && ls -la /content/da3/test.jpg /content/da3-small.onnx /content/da3-small-fixed.onnx


In [ ]:
# Square 504x504 image for the fixed-shape TRT engine (aspect-preserving
# preprocess maps square -> exactly the square engine shape).
!python3 -c "import cv2; im=cv2.imread('/content/da3/test.jpg'); print(im.shape); sq=cv2.resize(im,(504,504)); cv2.imwrite('/content/da3/test504.png',sq)"


In [ ]:
!source /content/da3_gate_env.sh && cd /content/da3 && MODEL=/content/da3-small.onnx IMAGE=/content/da3/test.jpg TRT_MODEL=/content/da3-small-fixed.onnx TRT_IMAGE=/content/da3/test504.png ./scripts/run_gpu_gate.sh /content/gate-results 2>&1 | tee /content/gate.log


In [ ]:
!cat /content/gate-results/STATUS; echo ---; cat /content/gate-results/environment.txt | head -20
# Copy /content/gate-results + output_depth*.jpg to Drive, then Runtime -> Disconnect and delete runtime.
